In [1]:
import re
import time
import requests
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
from google.colab import files

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

In [2]:
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Uploaded: []


In [3]:
remaining = pd.read_csv("remaining_101_files.csv")
events_clean = pd.read_csv("events_clean.csv")

print("Remaining rows:", len(remaining))
print("events_clean rows:", len(events_clean))

display(remaining.head())
display(events_clean.head())

Remaining rows: 101
events_clean rows: 2955


,ticker,filing_date,filing_type,accession_number,mda_filename,word_count_actual,digit_ratio_full,good_start_detected,suspicious_start_detected,contains_market_risk_start,...,first_400_chars,needs_manual_review,review_reason,manual_label,notes,notes_final,strict_label,strict_notes,original_manual_label,original_notes
0,AAPL,2019-10-31,10-K,0000320193-19-000119,AAPL_20191031_10-K_000032019319000119.txt,1018,0.015798,False,True,True,...,A. Quantitative and Qualitative Disclosures Ab...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN
1,AAPL,2020-10-30,10-K,0000320193-20-000096,AAPL_20201030_10-K_000032019320000096.txt,1018,0.015847,False,True,True,...,A. Quantitative and Qualitative Disclosures...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN
2,AAPL,2021-10-29,10-K,0000320193-21-000105,AAPL_20211029_10-K_000032019321000105.txt,1018,0.015847,False,True,True,...,A. Quantitative and Qualitative Disclosures...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN
3,AAPL,2022-10-28,10-K,0000320193-22-000108,AAPL_20221028_10-K_000032019322000108.txt,1017,0.015720,False,True,True,...,A. Quantitative and Qualitative Disclosures...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN
4,ABT,2023-02-17,10-K,0001628280-23-004026,ABT_20230217_10-K_000162828023004026.txt,723,0.060734,False,True,True,...,A. QUANTITATIVE AND QUALITATIVE DISCLOSURES AB...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN


,ticker,cik,filing_date,filing_type,accession_number,year,quarter
0,AAPL,320193,2019-01-30,10-Q,0000320193-19-000010,2019,1
1,AAPL,320193,2019-05-01,10-Q,0000320193-19-000066,2019,2
2,AAPL,320193,2019-07-31,10-Q,0000320193-19-000076,2019,3
3,AAPL,320193,2019-10-31,10-K,0000320193-19-000119,2019,4
4,AAPL,320193,2020-01-29,10-Q,0000320193-20-000010,2020,1


In [4]:
for df in [remaining, events_clean]:
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["filing_type"] = df["filing_type"].astype(str).str.strip().str.upper()
    df["accession_number"] = df["accession_number"].astype(str).str.strip()

remaining["filing_date"] = pd.to_datetime(
    remaining["filing_date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

events_clean["filing_date"] = pd.to_datetime(
    events_clean["filing_date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

if "manual_label" in remaining.columns:
    remaining["manual_label"] = remaining["manual_label"].astype(str).str.strip().str.lower()

In [5]:
repair_df = remaining.merge(
    events_clean[["ticker", "cik", "filing_date", "filing_type", "accession_number"]],
    on=["ticker", "filing_date", "filing_type", "accession_number"],
    how="left"
)

print("Rows after merge:", len(repair_df))
print("Missing cik:", repair_df["cik"].isna().sum())
display(repair_df.head())

Rows after merge: 101
Missing cik: 0


,ticker,filing_date,filing_type,accession_number,mda_filename,word_count_actual,digit_ratio_full,good_start_detected,suspicious_start_detected,contains_market_risk_start,...,needs_manual_review,review_reason,manual_label,notes,notes_final,strict_label,strict_notes,original_manual_label,original_notes,cik
0,AAPL,2019-10-31,10-K,0000320193-19-000119,AAPL_20191031_10-K_000032019319000119.txt,1018,0.015798,False,True,True,...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN,320193
1,AAPL,2020-10-30,10-K,0000320193-20-000096,AAPL_20201030_10-K_000032019320000096.txt,1018,0.015847,False,True,True,...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN,320193
2,AAPL,2021-10-29,10-K,0000320193-21-000105,AAPL_20211029_10-K_000032019321000105.txt,1018,0.015847,False,True,True,...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN,320193
3,AAPL,2022-10-28,10-K,0000320193-22-000108,AAPL_20221028_10-K_000032019322000108.txt,1017,0.015720,False,True,True,...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN,320193
4,ABT,2023-02-17,10-K,0001628280-23-004026,ABT_20230217_10-K_000162828023004026.txt,723,0.060734,False,True,True,...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN,1800


In [6]:
BASE_DIR = Path("/content/remaining_101_repair")
OUT_DIR = BASE_DIR / "repaired_txt"
BASE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPAIR_LOG_CSV = BASE_DIR / "remaining_101_repair_log.csv"

print("Output folder:", OUT_DIR)

Output folder: /content/remaining_101_repair/repaired_txt


In [7]:
HEADERS = {
    "User-Agent": "Cardiff University Student abhishekjc23@gmail.com",
    "Accept-Encoding": "gzip, deflate",
    "Host": "www.sec.gov"
}

def cik_nolead(cik):
    if pd.isna(cik):
        return None
    return str(int(float(cik)))

def acc_nodash(acc):
    return str(acc).replace("-", "")

def filing_base_url(cik, accession_number):
    return f"https://www.sec.gov/Archives/edgar/data/{cik_nolead(cik)}/{acc_nodash(accession_number)}"

def index_json_url(cik, accession_number):
    return filing_base_url(cik, accession_number) + "/index.json"

def get_index_json(cik, accession_number):
    url = index_json_url(cik, accession_number)
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

def safe_int(value, default=0):
    try:
        if value is None:
            return default
        value = str(value).strip()
        if value == "":
            return default
        return int(value)
    except Exception:
        return default

In [8]:
def choose_primary_doc(index_json, filing_type):
    try:
        items = index_json["directory"]["item"]
    except Exception:
        return None

    docs = []
    for x in items:
        name = str(x.get("name", ""))
        lower = name.lower()
        if lower.endswith((".htm", ".html", ".txt")):
            docs.append(x)

    if not docs:
        return None

    bad_words = ["ex-", "exhibit", "xbrl", "xml", "graphic", "image", "zip", "def14a", "8-k"]
    filtered = []
    for d in docs:
        nm = str(d.get("name", "")).lower()
        if any(b in nm for b in bad_words):
            continue
        filtered.append(d)

    if not filtered:
        filtered = docs

    ft = filing_type.lower().replace("-", "").replace("_", "")
    filing_pref = []
    for d in filtered:
        nm = str(d.get("name", "")).lower().replace("-", "").replace("_", "")
        if ft in nm:
            filing_pref.append(d)
        elif filing_type == "10-K" and "10k" in nm:
            filing_pref.append(d)
        elif filing_type == "10-Q" and "10q" in nm:
            filing_pref.append(d)

    if filing_pref:
        filtered = filing_pref

    filtered = sorted(filtered, key=lambda x: safe_int(x.get("size", 0), default=0), reverse=True)
    return filtered[0]["name"] if filtered else None

In [9]:
def download_filing_doc(cik, accession_number, filename):
    url = filing_base_url(cik, accession_number) + f"/{filename}"
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text

def html_to_text(html):
    soup = BeautifulSoup(html, "lxml")

    for tag in soup(["script", "style", "ix:header", "header", "footer"]):
        tag.decompose()

    text = soup.get_text("\n")
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    return text.strip()

def normalize_search_text(text):
    t = text.replace("\xa0", " ")
    t = t.replace("’", "'")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n+", "\n", t)
    return t

In [10]:
def find_heading_candidates(text, filing_type):
    t = normalize_search_text(text)
    low = t.lower()

    if filing_type == "10-K":
        start_pats = [
            r"item\s*7[\.\-:\s]+management[' ]s discussion and analysis of financial condition and results of operations",
            r"item\s*7[\.\-:\s]+management[' ]s discussion and analysis",
        ]
        end_pats = [
            r"item\s*7a[\.\-:\s]+quantitative and qualitative disclosures about market risk",
            r"item\s*8[\.\-:\s]+financial statements",
        ]
    else:
        start_pats = [
            r"item\s*2[\.\-:\s]+management[' ]s discussion and analysis of financial condition and results of operations",
            r"item\s*2[\.\-:\s]+management[' ]s discussion and analysis",
        ]
        end_pats = [
            r"item\s*3[\.\-:\s]+quantitative and qualitative disclosures about market risk",
            r"item\s*4[\.\-:\s]+controls and procedures",
        ]

    start_candidates = []
    for pat in start_pats:
        for m in re.finditer(pat, low, flags=re.I):
            start_candidates.append(m.start())

    end_candidates = []
    for pat in end_pats:
        for m in re.finditer(pat, low, flags=re.I):
            end_candidates.append(m.start())

    return sorted(set(start_candidates)), sorted(set(end_candidates)), t

In [11]:
def choose_best_section(text, filing_type):
    start_candidates, end_candidates, t = find_heading_candidates(text, filing_type)

    if not start_candidates:
        return None, None, "NO_START"
    if not end_candidates:
        return None, None, "NO_END"

    doc_len = len(t)
    best = None
    best_score = -10**9

    for s in start_candidates:
        valid_ends = [e for e in end_candidates if e > s]
        if not valid_ends:
            continue

        e = valid_ends[0]
        span = e - s

        if span < 3000 or span > 400000:
            continue

        start_chunk = t[max(0, s-300): min(doc_len, s+800)].lower()

        penalty = 0
        if "table of contents" in start_chunk:
            penalty -= 1000
        if "under the heading" in start_chunk:
            penalty -= 900
        if "refer to item" in start_chunk or "see item" in start_chunk:
            penalty -= 900
        if "part i" in start_chunk and "item 1" in start_chunk and "business" in start_chunk:
            penalty -= 1200

        position_score = s / max(doc_len, 1)
        span_score = -abs(span - 50000) / 50000

        score = penalty + (position_score * 200) + (span_score * 50)

        if score > best_score:
            best_score = score
            best = (s, e)

    if best is None:
        return None, None, "NO_VALID_SPAN"

    return best[0], best[1], "OK"

In [12]:
def extract_mda_ultra(text, filing_type):
    t = normalize_search_text(text)
    s, e, status = choose_best_section(t, filing_type)

    if status != "OK":
        return None, status

    mda_text = t[s:e].strip()
    wc = len(re.findall(r"\b[a-zA-Z]+\b", mda_text))

    if wc < 800:
        return None, "TOO_SHORT"

    first_400 = re.sub(r"\s+", " ", mda_text[:400]).lower()

    if filing_type == "10-K":
        good = (
            ("item 7" in first_400 and "management's discussion" in first_400) or
            ("item 7" in first_400 and "management’s discussion" in first_400)
        )
    else:
        good = (
            ("item 2" in first_400 and "management's discussion" in first_400) or
            ("item 2" in first_400 and "management’s discussion" in first_400)
        )

    if not good:
        return None, "BAD_START"

    if "table of contents" in first_400 or "under the heading" in first_400:
        return None, "BAD_START"

    return mda_text, "OK"

In [13]:
def repair_one_filing_ultra(row, sleep_seconds=0.2):
    ticker = row["ticker"]
    cik = row["cik"]
    filing_date = row["filing_date"]
    filing_type = row["filing_type"]
    accession_number = row["accession_number"]

    out = {
        "ticker": ticker,
        "filing_date": filing_date,
        "filing_type": filing_type,
        "accession_number": accession_number,
        "original_manual_label": row.get("manual_label", ""),
        "repair_status": "",
        "primary_doc": "",
        "repaired_filename": ""
    }

    try:
        if pd.isna(cik):
            out["repair_status"] = "NO_CIK"
            return out

        idx = get_index_json(cik, accession_number)
        primary_doc = choose_primary_doc(idx, filing_type)

        if primary_doc is None:
            out["repair_status"] = "NO_PRIMARY_DOC"
            return out

        out["primary_doc"] = primary_doc

        html = download_filing_doc(cik, accession_number, primary_doc)
        text = html_to_text(html)

        mda_text, status = extract_mda_ultra(text, filing_type)

        if status != "OK":
            out["repair_status"] = status
            return out

        repaired_filename = f"{ticker}_{filing_date}_{filing_type}_{accession_number}_ULTRA.txt"
        repaired_path = OUT_DIR / repaired_filename

        with open(repaired_path, "w", encoding="utf-8") as f:
            f.write(mda_text)

        out["repair_status"] = "OK"
        out["repaired_filename"] = repaired_filename

        time.sleep(sleep_seconds)
        return out

    except Exception as e:
        out["repair_status"] = f"ERROR: {str(e)[:120]}"
        return out

In [14]:
ultra_results = []

for i, row in repair_df.iterrows():
    result = repair_one_filing_ultra(row)
    ultra_results.append(result)

    if (i + 1) % 10 == 0:
        print(f"Processed {i+1}/{len(repair_df)}")

ultra_log = pd.DataFrame(ultra_results)
ultra_log.to_csv(REPAIR_LOG_CSV, index=False)

print("Ultra repair complete.")
display(ultra_log["repair_status"].value_counts())
display(ultra_log.head())

Processed 10/101
Processed 20/101
Processed 30/101
Processed 40/101
Processed 50/101
Processed 60/101
Processed 70/101
Processed 80/101
Processed 90/101
Processed 100/101
Ultra repair complete.


,count
repair_status,
OK,54
NO_START,41
NO_VALID_SPAN,3
BAD_START,3


,ticker,filing_date,filing_type,accession_number,original_manual_label,repair_status,primary_doc,repaired_filename
0,AAPL,2019-10-31,10-K,0000320193-19-000119,incorrect,OK,a10-k20199282019.htm,AAPL_2019-10-31_10-K_0000320193-19-000119_ULTR...
1,AAPL,2020-10-30,10-K,0000320193-20-000096,incorrect,OK,aapl-20200926.htm,AAPL_2020-10-30_10-K_0000320193-20-000096_ULTR...
2,AAPL,2021-10-29,10-K,0000320193-21-000105,incorrect,OK,aapl-20210925.htm,AAPL_2021-10-29_10-K_0000320193-21-000105_ULTR...
3,AAPL,2022-10-28,10-K,0000320193-22-000108,incorrect,OK,aapl-20220924.htm,AAPL_2022-10-28_10-K_0000320193-22-000108_ULTR...
4,ABT,2023-02-17,10-K,0001628280-23-004026,incorrect,NO_START,abt-20221231x10kexx21.htm,


In [15]:
def preview_ultra_file(filename, start_chars=2500, end_chars=1200):
    path = OUT_DIR / filename
    if not path.exists():
        print("File not found:", path)
        return

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    print("=" * 120)
    print("FILE:", filename)
    print("=" * 120)
    print("\n--- START PREVIEW ---\n")
    print(text[:start_chars])
    print("\n--- END PREVIEW ---\n")
    print(text[-end_chars:])
    print("\nWord count:", len(re.findall(r"\b[a-zA-Z]+\b", text)))
    print("=" * 120)

In [16]:
ultra_ok = ultra_log.loc[ultra_log["repair_status"] == "OK"].copy()

print("Ultra recovered OK files:", len(ultra_ok))
display(ultra_ok.head(20))

Ultra recovered OK files: 54


,ticker,filing_date,filing_type,accession_number,original_manual_label,repair_status,primary_doc,repaired_filename
0,AAPL,2019-10-31,10-K,0000320193-19-000119,incorrect,OK,a10-k20199282019.htm,AAPL_2019-10-31_10-K_0000320193-19-000119_ULTR...
1,AAPL,2020-10-30,10-K,0000320193-20-000096,incorrect,OK,aapl-20200926.htm,AAPL_2020-10-30_10-K_0000320193-20-000096_ULTR...
2,AAPL,2021-10-29,10-K,0000320193-21-000105,incorrect,OK,aapl-20210925.htm,AAPL_2021-10-29_10-K_0000320193-21-000105_ULTR...
3,AAPL,2022-10-28,10-K,0000320193-22-000108,incorrect,OK,aapl-20220924.htm,AAPL_2022-10-28_10-K_0000320193-22-000108_ULTR...
6,AMT,2023-02-23,10-K,0001053507-23-000023,incorrect,OK,amt-20221231.htm,AMT_2023-02-23_10-K_0001053507-23-000023_ULTRA...
7,AMZN,2019-10-25,10-Q,0001018724-19-000089,incorrect,OK,amzn-2019930x10q.htm,AMZN_2019-10-25_10-Q_0001018724-19-000089_ULTR...
8,APH,2021-07-30,10-Q,0001558370-21-009700,partial,OK,aph-20210630x10q.htm,APH_2021-07-30_10-Q_0001558370-21-009700_ULTRA...
9,APH,2021-10-29,10-Q,0001558370-21-013825,partial,OK,aph-20210930x10q.htm,APH_2021-10-29_10-Q_0001558370-21-013825_ULTRA...
10,APH,2022-02-09,10-K,0001558370-22-000961,partial,OK,aph-20211231x10k.htm,APH_2022-02-09_10-K_0001558370-22-000961_ULTRA...
11,APH,2022-07-29,10-Q,0001558370-22-011379,partial,OK,aph-20220630x10q.htm,APH_2022-07-29_10-Q_0001558370-22-011379_ULTRA...


In [17]:
ok_ultra_df = ultra_log.loc[
    ultra_log["repair_status"] == "OK",
    ["ticker", "filing_date", "filing_type", "accession_number", "repaired_filename"]
].dropna().copy()

print("Ultra OK files:", len(ok_ultra_df))
display(ok_ultra_df.head())

Ultra OK files: 54


,ticker,filing_date,filing_type,accession_number,repaired_filename
0,AAPL,2019-10-31,10-K,0000320193-19-000119,AAPL_2019-10-31_10-K_0000320193-19-000119_ULTR...
1,AAPL,2020-10-30,10-K,0000320193-20-000096,AAPL_2020-10-30_10-K_0000320193-20-000096_ULTR...
2,AAPL,2021-10-29,10-K,0000320193-21-000105,AAPL_2021-10-29_10-K_0000320193-21-000105_ULTR...
3,AAPL,2022-10-28,10-K,0000320193-22-000108,AAPL_2022-10-28_10-K_0000320193-22-000108_ULTR...
6,AMT,2023-02-23,10-K,0001053507-23-000023,AMT_2023-02-23_10-K_0001053507-23-000023_ULTRA...


In [18]:
def read_text_safe(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def first_chunk(text, n=1500):
    return re.sub(r"\s+", " ", text[:n]).strip()

def word_count_alpha(text):
    return len(re.findall(r"\b[a-zA-Z]+\b", text))

def validate_ultra_start(text, filing_type):
    t = first_chunk(text, 1500).lower()
    t = t.replace("’", "'")

    if filing_type == "10-K":
        strong_heading = bool(re.search(
            r"item\s*7[\.\-:\s]+management[' ]s discussion and analysis of financial condition and results of operations",
            t
        ))
        weak_heading = bool(re.search(
            r"management[' ]s discussion and analysis of financial condition and results of operations",
            t
        ))
    else:  # 10-Q
        strong_heading = bool(re.search(
            r"item\s*2[\.\-:\s]+management[' ]s discussion and analysis of financial condition and results of operations",
            t
        ))
        weak_heading = bool(re.search(
            r"management[' ]s discussion and analysis of financial condition and results of operations",
            t
        ))

    bad_reference = (
        "under the heading" in t or
        "see part ii, item 7" in t or
        "see item 2" in t or
        "refer to item" in t or
        "table of contents" in t or
        "part i item 1" in t or
        "company background" in t or
        ("business" in t[:400] and "item 1" in t[:400])
    )

    toc_like = (
        ("item 7a" in t[:500] and "item 8" in t[:500]) or
        ("item 3" in t[:500] and "item 4" in t[:500])
    )

    return {
        "strong_heading_match": strong_heading,
        "weak_heading_match": weak_heading,
        "bad_reference_start": bad_reference,
        "toc_like_start": toc_like,
        "first_300_chars": text[:300].replace("\n", " ")
    }

In [20]:
records = []

for _, row in ok_ultra_df.iterrows():
    path = OUT_DIR / row["repaired_filename"]
    text = read_text_safe(path)
    checks = validate_ultra_start(text, row["filing_type"])

    records.append({
        "ticker": row["ticker"],
        "filing_date": row["filing_date"],
        "filing_type": row["filing_type"],
        "accession_number": row["accession_number"],
        "repaired_filename": row["repaired_filename"],
        "word_count": word_count_alpha(text),
        **checks
    })

ultra_validation_df = pd.DataFrame(records)
display(ultra_validation_df.head())

,ticker,filing_date,filing_type,accession_number,repaired_filename,word_count,strong_heading_match,weak_heading_match,bad_reference_start,toc_like_start,first_300_chars
0,AAPL,2019-10-31,10-K,0000320193-19-000119,AAPL_2019-10-31_10-K_0000320193-19-000119_ULTR...,4715,True,True,True,False,Item 7. Management's Discussion and Analysis o...
1,AAPL,2020-10-30,10-K,0000320193-20-000096,AAPL_2020-10-30_10-K_0000320193-20-000096_ULTR...,4463,True,True,False,False,Item 7. Management's Discussion and Analysis o...
2,AAPL,2021-10-29,10-K,0000320193-21-000105,AAPL_2021-10-29_10-K_0000320193-21-000105_ULTR...,2609,True,True,False,False,Item 7. Management's Discussion and Analysis o...
3,AAPL,2022-10-28,10-K,0000320193-22-000108,AAPL_2022-10-28_10-K_0000320193-22-000108_ULTR...,2417,True,True,False,False,Item 7. Management's Discussion and Analysis o...
4,AMT,2023-02-23,10-K,0001053507-23-000023,AMT_2023-02-23_10-K_0001053507-23-000023_ULTRA...,16253,True,True,False,False,ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS O...


In [21]:
ultra_validation_df["needs_manual_review"] = (
    (~ultra_validation_df["strong_heading_match"]) |
    (ultra_validation_df["bad_reference_start"]) |
    (ultra_validation_df["toc_like_start"])
)

likely_good_ultra = ultra_validation_df.loc[~ultra_validation_df["needs_manual_review"]].copy()
needs_review_ultra = ultra_validation_df.loc[ultra_validation_df["needs_manual_review"]].copy()

print("Likely good ultra files:", len(likely_good_ultra))
print("Ultra files needing manual review:", len(needs_review_ultra))

Likely good ultra files: 40
Ultra files needing manual review: 14


In [22]:
likely_good_ultra["manual_label"] = ""
likely_good_ultra["notes"] = ""

needs_review_ultra["manual_label"] = ""
needs_review_ultra["notes"] = ""

likely_good_ultra.to_csv("ultra_repaired_files_likely_good.csv", index=False)
needs_review_ultra.to_csv("ultra_repaired_files_needing_review.csv", index=False)

print("Saved ultra_repaired_files_likely_good.csv")
print("Saved ultra_repaired_files_needing_review.csv")

Saved ultra_repaired_files_likely_good.csv
Saved ultra_repaired_files_needing_review.csv


In [23]:
def preview_ultra_file(filename, start_chars=2500, end_chars=1200):
    path = OUT_DIR / filename
    if not path.exists():
        print("File not found:", path)
        return

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    print("=" * 120)
    print("FILE:", filename)
    print("=" * 120)
    print("\n--- START PREVIEW ---\n")
    print(text[:start_chars])
    print("\n--- END PREVIEW ---\n")
    print(text[-end_chars:])
    print("\nWord count:", len(re.findall(r"\b[a-zA-Z]+\b", text)))
    print("=" * 120)

In [24]:
ultra_likely = pd.read_csv("ultra_repaired_files_likely_good.csv")

ultra_likely["ticker"] = ultra_likely["ticker"].astype(str).str.strip().str.upper()
ultra_likely["filing_type"] = ultra_likely["filing_type"].astype(str).str.strip().str.upper()
ultra_likely["accession_number"] = ultra_likely["accession_number"].astype(str).str.strip()
ultra_likely["filing_date"] = pd.to_datetime(
    ultra_likely["filing_date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

ultra_likely["manual_label"] = ultra_likely.get("manual_label", "").fillna("").astype(str).str.strip().str.lower()
ultra_likely["notes"] = ultra_likely.get("notes", "").fillna("").astype(str)

correct_keys_ultra = {
    ("AAPL", "2020-10-30", "10-K"),
    ("AAPL", "2021-10-29", "10-K"),
    ("AAPL", "2022-10-28", "10-K"),

    ("AMT", "2023-02-23", "10-K"),
    ("AMZN", "2019-10-25", "10-Q"),

    ("APH", "2022-02-09", "10-K"),
    ("APH", "2023-02-08", "10-K"),
    ("APH", "2024-02-07", "10-K"),

    ("CB", "2023-02-24", "10-K"),

    ("CVS", "2024-02-07", "10-K"),

    ("EQIX", "2020-02-21", "10-K"),
    ("EQIX", "2021-02-19", "10-K"),
    ("EQIX", "2022-02-18", "10-K"),
    ("EQIX", "2023-02-17", "10-K"),
    ("EQIX", "2024-02-16", "10-K"),

    ("GD", "2020-01-29", "10-K"),
    ("GD", "2021-02-03", "10-K"),

    ("GILD", "2022-08-08", "10-Q"),
    ("GILD", "2022-11-02", "10-Q"),
    ("GILD", "2023-05-03", "10-Q"),
    ("GILD", "2023-08-04", "10-Q"),
    ("GILD", "2023-11-07", "10-Q"),
    ("GILD", "2024-05-08", "10-Q"),
    ("GILD", "2024-08-08", "10-Q"),
    ("GILD", "2024-11-12", "10-Q"),

    ("ISRG", "2019-02-04", "10-K"),
    ("ISRG", "2020-02-07", "10-K"),
    ("ISRG", "2021-02-10", "10-K"),
    ("ISRG", "2022-02-03", "10-K"),
    ("ISRG", "2023-02-10", "10-K"),
    ("ISRG", "2024-01-31", "10-K"),

    ("JCI", "2019-02-01", "10-Q"),
    ("JCI", "2019-05-03", "10-Q"),

    ("MO", "2021-02-26", "10-K"),
    ("PANW", "2019-05-31", "10-Q"),
    ("PH", "2020-08-07", "10-K"),
}

ultra_likely.loc[
    ultra_likely.apply(
        lambda r: (r["ticker"], r["filing_date"], r["filing_type"]) in correct_keys_ultra,
        axis=1
    ),
    ["manual_label", "notes"]
] = ["correct", "accepted after strict ultra-file review"]

print(ultra_likely["manual_label"].value_counts(dropna=False))
display(ultra_likely.loc[ultra_likely["manual_label"] == "correct"].head(40))

manual_label
correct    32
            8
Name: count, dtype: int64


,ticker,filing_date,filing_type,accession_number,repaired_filename,word_count,strong_heading_match,weak_heading_match,bad_reference_start,toc_like_start,first_300_chars,needs_manual_review,manual_label,notes
0,AAPL,2020-10-30,10-K,0000320193-20-000096,AAPL_2020-10-30_10-K_0000320193-20-000096_ULTR...,4463,True,True,False,False,Item 7. Management's Discussion and Analysis o...,False,correct,accepted after strict ultra-file review
1,AAPL,2021-10-29,10-K,0000320193-21-000105,AAPL_2021-10-29_10-K_0000320193-21-000105_ULTR...,2609,True,True,False,False,Item 7. Management's Discussion and Analysis o...,False,correct,accepted after strict ultra-file review
2,AAPL,2022-10-28,10-K,0000320193-22-000108,AAPL_2022-10-28_10-K_0000320193-22-000108_ULTR...,2417,True,True,False,False,Item 7. Management's Discussion and Analysis o...,False,correct,accepted after strict ultra-file review
3,AMT,2023-02-23,10-K,0001053507-23-000023,AMT_2023-02-23_10-K_0001053507-23-000023_ULTRA...,16253,True,True,False,False,ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS O...,False,correct,accepted after strict ultra-file review
4,AMZN,2019-10-25,10-Q,0001018724-19-000089,AMZN_2019-10-25_10-Q_0001018724-19-000089_ULTR...,6253,True,True,False,False,Item 2. Management's Discussion and Analysis o...,False,correct,accepted after strict ultra-file review
5,APH,2022-02-09,10-K,0001558370-22-000961,APH_2022-02-09_10-K_0001558370-22-000961_ULTRA...,15011,True,True,False,False,Item 7. Management's Discussion and Analysis o...,False,correct,accepted after strict ultra-file review
6,APH,2023-02-08,10-K,0001558370-23-001036,APH_2023-02-08_10-K_0001558370-23-001036_ULTRA...,15814,True,True,False,False,Item 7. Management's Discussion and Analysis o...,False,correct,accepted after strict ultra-file review
7,APH,2024-02-07,10-K,0001558370-24-000866,APH_2024-02-07_10-K_0001558370-24-000866_ULTRA...,14478,True,True,False,False,Item 7. Management's Discussion and Analysis o...,False,correct,accepted after strict ultra-file review
8,CB,2023-02-24,10-K,0000896159-23-000007,CB_2023-02-24_10-K_0000896159-23-000007_ULTRA.txt,26435,True,True,False,False,ITEM 7. Management's Discussion and Analysis o...,False,correct,accepted after strict ultra-file review
13,CVS,2024-02-07,10-K,0000064803-24-000007,CVS_2024-02-07_10-K_0000064803-24-000007_ULTRA...,15734,True,True,False,False,Item 7. Management's Discussion and Analysis o...,False,correct,accepted after strict ultra-file review


In [25]:
ULTRA_CORRECT_CSV = "/content/ultra_correct_36_files.csv"

ultra_correct = ultra_likely.loc[ultra_likely["manual_label"] == "correct"].copy()
ultra_correct.to_csv(ULTRA_CORRECT_CSV, index=False)

print("Ultra-correct rows:", len(ultra_correct))
print("Saved:", ULTRA_CORRECT_CSV)

Ultra-correct rows: 32
Saved: /content/ultra_correct_36_files.csv


In [26]:
from pathlib import Path
import shutil
import zipfile
from google.colab import files

ULTRA_DIR = Path("/content/remaining_101_repair/repaired_txt")
ULTRA_DOWNLOAD_DIR = Path("/content/ultra_correct_36_txt")
ULTRA_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# clear old files if any
for old_file in ULTRA_DOWNLOAD_DIR.glob("*.txt"):
    old_file.unlink()

copied = 0
missing = []

for _, row in ultra_correct.iterrows():
    fname = row["repaired_filename"]
    src = ULTRA_DIR / fname
    dst = ULTRA_DOWNLOAD_DIR / fname

    if src.exists():
        shutil.copy2(src, dst)
        copied += 1
    else:
        missing.append(fname)

print("Copied txt files:", copied)
print("Missing txt files:", len(missing))
if missing:
    print("Missing list:")
    for x in missing:
        print(x)

Copied txt files: 32
Missing txt files: 0


In [27]:
ULTRA_CORRECT_ZIP = "/content/ultra_correct_36_txt.zip"

with zipfile.ZipFile(ULTRA_CORRECT_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    for txt_file in ULTRA_DOWNLOAD_DIR.glob("*.txt"):
        zipf.write(txt_file, arcname=txt_file.name)

print("Saved zip:", ULTRA_CORRECT_ZIP)

Saved zip: /content/ultra_correct_36_txt.zip


In [29]:
import pandas as pd

remaining_df = pd.read_csv("remaining_101_files.csv")
ultra_correct = pd.read_csv("ultra_correct_36_files.csv")

print("remaining_101 rows:", len(remaining_df))
print("ultra_correct rows:", len(ultra_correct))

remaining_101 rows: 101
ultra_correct rows: 32


In [30]:
for df in [remaining_df, ultra_correct]:
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["filing_type"] = df["filing_type"].astype(str).str.strip().str.upper()
    df["accession_number"] = df["accession_number"].astype(str).str.strip()
    df["filing_date"] = pd.to_datetime(df["filing_date"], errors="coerce").dt.strftime("%Y-%m-%d")

In [31]:
ultra_correct_keys = set(
    zip(
        ultra_correct["ticker"],
        ultra_correct["filing_date"],
        ultra_correct["filing_type"],
        ultra_correct["accession_number"]
    )
)

print("Accepted ultra keys:", len(ultra_correct_keys))

Accepted ultra keys: 32


In [32]:
remaining_65 = remaining_df.loc[
    ~remaining_df.apply(
        lambda r: (r["ticker"], r["filing_date"], r["filing_type"], r["accession_number"]) in ultra_correct_keys,
        axis=1
    )
].copy()

print("Remaining unresolved after accepting 36:", len(remaining_65))
display(remaining_65.head())

Remaining unresolved after accepting 36: 69


,ticker,filing_date,filing_type,accession_number,mda_filename,word_count_actual,digit_ratio_full,good_start_detected,suspicious_start_detected,contains_market_risk_start,...,first_400_chars,needs_manual_review,review_reason,manual_label,notes,notes_final,strict_label,strict_notes,original_manual_label,original_notes
0,AAPL,2019-10-31,10-K,0000320193-19-000119,AAPL_20191031_10-K_000032019319000119.txt,1018,0.015798,False,True,True,...,A. Quantitative and Qualitative Disclosures Ab...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN
4,ABT,2023-02-17,10-K,0001628280-23-004026,ABT_20230217_10-K_000162828023004026.txt,723,0.060734,False,True,True,...,A. QUANTITATIVE AND QUALITATIVE DISCLOSURES AB...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN
5,ABT,2024-02-16,10-K,0001628280-24-005348,ABT_20240216_10-K_000162828024005348.txt,733,0.059546,False,True,True,...,A. QUANTITATIVE AND QUALITATIVE DISCLOSURES AB...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN
8,APH,2021-07-30,10-Q,0001558370-21-009700,APH_20210730_10-Q_000155837021009700.txt,9495,0.037938,False,False,False,...,are specifically to our continuing operations ...,True,flagged,partial,appears to start slightly after heading/introd...,NaN,partial,appears to start slightly after heading/introd...,NaN,NaN
9,APH,2021-10-29,10-Q,0001558370-21-013825,APH_20211029_10-Q_000155837021013825.txt,9780,0.039108,False,False,False,...,are specifically to our continuing operations ...,True,flagged,partial,appears to start slightly after heading/introd...,NaN,partial,appears to start slightly after heading/introd...,NaN,NaN


In [33]:
REMAINING_65_CSV = "/content/remaining_65_files.csv"
remaining_65.to_csv(REMAINING_65_CSV, index=False)
print("Saved:", REMAINING_65_CSV)

Saved: /content/remaining_65_files.csv
